In [ ]:
# Setup
import sys
from pathlib import Path
import numpy as np
import json
import cv2

sys.path.insert(0, str(Path.cwd().parent))

DATA_DIR = Path(r'D:\sim-bench\results\Google_Germany')
TEST_FACES = [569, 577, 573, 553, 550, 545]
crops_dir = DATA_DIR / 'face_crops'
embeddings_path = DATA_DIR / 'embeddings_FRESH_2026-03-17_01-00-27.npy'
metadata_path = DATA_DIR / 'benchmark_2026-03-01_01-10-04.json'

print(f"Testing faces: {TEST_FACES}")

In [ ]:
# Method 1: Load from NPY
def load_npy_embedding(face_id):
    """Load embedding for one face from .npy file."""
    # Load once and cache
    if not hasattr(load_npy_embedding, '_cache'):
        stored_array = np.load(embeddings_path)
        with open(metadata_path) as f:
            metadata = json.load(f)
        face_ids = [i if meta.get('face_index') is None else meta.get('face_index') 
                    for i, meta in enumerate(metadata['face_metadata'])]
        load_npy_embedding._cache = {fid: stored_array[i] for i, fid in enumerate(face_ids)}
    
    return load_npy_embedding._cache[face_id]

In [ ]:
# Method 2: Fresh extraction (face_cluster)
def extract_fresh_embedding(face_id):
    """Extract embedding using face_cluster.InsightFaceEmbedder."""
    from face_cluster import InsightFaceEmbedder
    
    # Load model once and cache
    if not hasattr(extract_fresh_embedding, '_embedder'):
        extract_fresh_embedding._embedder = InsightFaceEmbedder(model_name='buffalo_l', ctx_id=-1)
    
    img_path = crops_dir / f'face_{face_id:04d}_aligned.jpg'
    img = cv2.imread(str(img_path))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    return extract_fresh_embedding._embedder.get_embedding(img_rgb)

In [ ]:
# Method 3: Pipeline extraction (benchmark method)
def extract_pipeline_embedding(face_id):
    """Extract embedding using InsightFaceNativeExtractor (benchmark method)."""
    from sim_bench.pipeline.face_embedding.insightface_native import InsightFaceNativeExtractor
    
    # Load model once and cache
    if not hasattr(extract_pipeline_embedding, '_extractor'):
        config = {"backend": "insightface", "device": "cpu", "model_name": "buffalo_l"}
        extract_pipeline_embedding._extractor = InsightFaceNativeExtractor(config)
    
    img_path = crops_dir / f'face_{face_id:04d}_aligned.jpg'
    img = cv2.imread(str(img_path))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    embeddings = extract_pipeline_embedding._extractor.extract_batch([img_rgb], [{'face_id': face_id}])
    return embeddings[0]

In [ ]:
# Method that calls all three
def get_all_embeddings(face_id):
    """Get embeddings from all three methods for a given face."""
    return {
        'npy': load_npy_embedding(face_id),
        'fresh': extract_fresh_embedding(face_id),
        'pipeline': extract_pipeline_embedding(face_id)
    }

In [ ]:
# Build comparison table
def cosine_distance(emb1, emb2):
    return 1 - np.dot(emb1, emb2) / (np.linalg.norm(emb1) * np.linalg.norm(emb2))

# Get all embeddings
all_embeddings = {face_id: get_all_embeddings(face_id) for face_id in TEST_FACES}

# Compare distances from target face
target = TEST_FACES[0]
print(f"\nDistances from face {target}:")
print(f"{'Neighbor':<10} {'NPY':<12} {'Fresh':<12} {'Pipeline':<12} {'NPY-Fresh':<12} {'NPY-Pipeline':<12}")
print("-"*80)

for neighbor in TEST_FACES[1:]:
    dist_npy = cosine_distance(all_embeddings[target]['npy'], all_embeddings[neighbor]['npy'])
    dist_fresh = cosine_distance(all_embeddings[target]['fresh'], all_embeddings[neighbor]['fresh'])
    dist_pipeline = cosine_distance(all_embeddings[target]['pipeline'], all_embeddings[neighbor]['pipeline'])
    
    diff_npy_fresh = abs(dist_npy - dist_fresh)
    diff_npy_pipeline = abs(dist_npy - dist_pipeline)
    
    print(f"{neighbor:<10} {dist_npy:<12.6f} {dist_fresh:<12.6f} {dist_pipeline:<12.6f} "
          f"{diff_npy_fresh:<12.6f} {diff_npy_pipeline:<12.6f}")

print("\nVERDICT:")
print("If NPY ≈ Pipeline (diff < 0.01): NPY embeddings are correct")
print("If Fresh ≈ Pipeline (diff < 0.01): Fresh method matches benchmark")
print("If all differ: Different preprocessing between methods")

In [ ]:
# Search all faces for stored[569]
all_face_ids = sorted(load_npy_embedding._cache.keys())
stored_569 = all_embeddings[569]['npy']

distances = {}
for fid in all_face_ids:
    try:
        fresh = extract_fresh_embedding(fid)
        distances[fid] = cosine_distance(stored_569, fresh)
    except:
        pass

# Top 10 closest matches
for fid, dist in sorted(distances.items(), key=lambda x: x[1])[:10]:
    print(f"{fid}: {dist:.6f}")

In [ ]:
# Verify pattern: stored[N] = fresh[N+2]
test_ids = [569, 573, 577, 553, 550, 545]
for fid in test_ids:
    stored_emb = all_embeddings[fid]['npy']
    fresh_emb = extract_fresh_embedding(fid + 2)
    dist = cosine_distance(stored_emb, fresh_emb)
    print(f"stored[{fid}] vs fresh[{fid+2}]: {dist:.6f}")

In [ ]:
# Check metadata mapping
with open(metadata_path) as f:
    metadata = json.load(f)

print("First 10 metadata entries (face_index):")
for i in range(10):
    face_idx = metadata['face_metadata'][i].get('face_index')
    print(f"Position {i}: face_index={face_idx}")

In [ ]:
# Check which crop files actually exist
crop_files = sorted(crops_dir.glob('face_*_aligned.jpg'))
print(f"Total crops: {len(crop_files)}")
print(f"First 10: {[f.stem for f in crop_files[:10]]}")
print(f"Around 569: {[f.stem for f in crop_files[567:573]]}")